# Practical 7: Regression and ANOVA

## Aim
Simple regressions and one-way ANOVA on the Teaching Ratings data, calculated with NumPy and SciPy.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
df = pd.read_csv("TeachingRatings.csv")

In [3]:
def ols(x, y):
    X = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    n, k = X.shape
    sse = (resid ** 2).sum()
    sst = ((y - y.mean()) ** 2).sum()
    r2 = 1 - sse / sst
    f = ((sst - sse) / (k - 1)) / (sse / (n - k))
    p = stats.f.sf(f, k - 1, n - k)
    return coef, r2, f, p

In [4]:
print("Q1. eval ~ gender")
x = (df["gender"] == "male").astype(float).values
coef, r2, f, p = ols(x, df["eval"].values)
print("Intercept (female mean): %.4f" % coef[0])
print("Male coefficient: %.4f" % coef[1])
print("R-squared: %.3f, F: %.2f, p-value: %.5f" % (r2, f, p))
print("Decision:", "Reject H0, gender is significant" if p < 0.05 else "Fail to reject H0")

Q1. eval ~ gender
Intercept (female mean): 3.9010
Male coefficient: 0.1680
R-squared: 0.022, F: 10.56, p-value: 0.00124
Decision: Reject H0, gender is significant


In [5]:
print("\nQ2. ANOVA: beauty ~ age group")
df["age_group"] = pd.cut(df["age"], [0, 40, 50, 100], labels=["Young", "Middle", "Old"])
y = df["beauty"]
grand = y.mean()
between = sum(len(g) * (g.mean() - grand) ** 2 for _, g in y.groupby(df["age_group"], observed=True))
within = sum(((g - g.mean()) ** 2).sum() for _, g in y.groupby(df["age_group"], observed=True))
dfb, dfw = 2, len(df) - 3
f = (between / dfb) / (within / dfw)
p = stats.f.sf(f, dfb, dfw)
print(pd.DataFrame({"df": [dfb, dfw], "sum_sq": [between, within],
                    "mean_sq": [between / dfb, within / dfw],
                    "F": [f, np.nan], "p": [p, np.nan]},
                   index=["age_group", "Residual"]).round(6).to_string())
print("p-value: %.3e" % p)
print("Decision:", "Reject H0, beauty differs by age group" if p < 0.05 else "Fail to reject H0")


Q2. ANOVA: beauty ~ age group
            df      sum_sq   mean_sq          F    p
age_group    2   17.800717  8.900359  15.189048  0.0
Residual   460  269.547179  0.585972        NaN  NaN
p-value: 4.094e-07
Decision: Reject H0, beauty differs by age group


In [6]:
print("\nQ3. eval ~ beauty")
coef, r2, f, p = ols(df["beauty"].values, df["eval"].values)
print("Intercept: %.4f" % coef[0])
print("Beauty coefficient: %.4f" % coef[1])
print("R-squared: %.3f, F: %.2f, p-value: %.6f" % (r2, f, p))
print("Decision:", "Reject H0, beauty is significant" if p < 0.05 else "Fail to reject H0")


Q3. eval ~ beauty
Intercept: 3.9983
Beauty coefficient: 0.1330
R-squared: 0.036, F: 17.08, p-value: 0.000042
Decision: Reject H0, beauty is significant


In [7]:
print("\nResult: male instructors score about 0.168 higher, beauty differs across age groups,")
print("and each unit of beauty adds about 0.133 to the evaluation score.")


Result: male instructors score about 0.168 higher, beauty differs across age groups,
and each unit of beauty adds about 0.133 to the evaluation score.
